<a href="https://colab.research.google.com/github/nishita-ds/Hyperparameter-tuning-using-Optuna/blob/main/optuna_basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 3.8 MB/s eta 0:00:00


In [20]:
# Import necessary libraries
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the Pima Indian Diabetes dataset from sklearn
# Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)csv
url = "https://raw.githubusercontent.com/liquidcarrot/data.pima-indians-diabetes/refs/heads/master/src/index.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,pregnancies,plasma glucose concentration,diastolic blood pressure,triceps skinfold thickness,insulin,body mass index,diabetes pedigree function,age,diabetic
1,6,148,72,35,0,33.6,0.627,50,1
2,1,85,66,29,0,26.6,0.351,31,0
3,8,183,64,0,0,23.3,0.672,32,1
4,1,89,66,23,94,28.1,0.167,21,0


In [21]:
import numpy as np
import pandas as pd

# Drop the first row which contains the original string header values
df = df.iloc[1:].copy()

# Convert all columns to numeric, coercing errors to NaN
# This is crucial because all data was read as strings due to the header row initially
for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
# Now df.mean() will work correctly as all columns are numeric
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [22]:
# Split into features (X) and target (y)
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into training and test sets (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [24]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-04-07 16:31:11,250] A new study created in memory with name: no-name-633d8411-c8ec-48ba-bd76-5b853102c1b1
[I 2026-04-07 16:31:11,808] Trial 0 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 110, 'max_depth': 3}. Best is trial 0 with value: 0.7597765363128491.
[I 2026-04-07 16:31:12,732] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 162, 'max_depth': 10}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-04-07 16:31:13,851] Trial 2 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 198, 'max_depth': 13}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-04-07 16:31:14,731] Trial 3 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 157, 'max_depth': 14}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-04-07 16:31:15,068] Trial 4 finished with value: 0.7635009310986964 and parameters: {'n_estimators': 59, 'max_depth': 15}. Best is trial 1 with value: 0.772811

In [25]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 54, 'max_depth': 7}


In [26]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


# Samplers in Optuna

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

# Random Search

In [11]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-04-07 16:19:48,183] A new study created in memory with name: no-name-50ee8910-971a-48f5-a28d-0b2dfe320412
[I 2026-04-07 16:19:49,439] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 58, 'max_depth': 12}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-07 16:19:50,292] Trial 1 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 59, 'max_depth': 6}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-07 16:19:51,523] Trial 2 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 87, 'max_depth': 8}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-07 16:19:52,695] Trial 3 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 84, 'max_depth': 20}. Best is trial 1 with value: 0.7709497206703911.
[I 2026-04-07 16:19:53,595] Trial 4 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 85, 'max_depth': 9}. Best is trial 1 with value: 0.770949720670

In [12]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 196, 'max_depth': 19}


In [13]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


# Grid Search

In [14]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [15]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-04-07 16:25:06,946] A new study created in memory with name: no-name-2c912add-3773-49c7-b911-fea03286b395
[I 2026-04-07 16:25:08,194] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-07 16:25:09,785] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-04-07 16:25:10,323] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-07 16:25:11,310] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-04-07 16:25:13,236] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [16]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7746741154562384
Best hyperparameters: {'n_estimators': 50, 'max_depth': 5}


In [17]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


# Optuna Visualization

In [27]:
# For visualization
from optuna.visualization import plot_optimization_history,plot_parallel_coordinate,plot_slice,plot_contour,plot_param_importances

In [28]:
# 1.Optimization History
plot_optimization_history(study).show()

In [29]:
# 2. Parallel coordinate plot
plot_parallel_coordinate(study).show()

In [30]:
# 3.Slice plot
plot_slice(study).show()

In [31]:
# 4.Contour plot
plot_contour(study).show()

In [32]:
# Hyperparameter Importances
plot_param_importances(study).show()

# Optimizing Multiple ML Models

In [33]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [34]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [35]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-04-07 16:47:04,951] A new study created in memory with name: no-name-7afcb41e-3fc3-480d-887e-8973f5930dd2
[I 2026-04-07 16:47:07,780] Trial 0 finished with value: 0.7728119180633147 and parameters: {'classifier': 'RandomForest', 'n_estimators': 214, 'max_depth': 18, 'min_samples_split': 10, 'min_samples_leaf': 9, 'bootstrap': False}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-07 16:47:07,926] Trial 1 finished with value: 0.7597765363128491 and parameters: {'classifier': 'SVM', 'C': 0.881277946095126, 'kernel': 'sigmoid', 'gamma': 'auto'}. Best is trial 0 with value: 0.7728119180633147.
[I 2026-04-07 16:47:07,972] Trial 2 finished with value: 0.7821229050279329 and parameters: {'classifier': 'SVM', 'C': 0.38091221405647246, 'kernel': 'linear', 'gamma': 'scale'}. Best is trial 2 with value: 0.7821229050279329.
[I 2026-04-07 16:47:09,153] Trial 3 finished with value: 0.7597765363128491 and parameters: {'classifier': 'RandomForest', 'n_estimators': 129, 'max_depth':

In [36]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11304208869622426, 'kernel': 'linear', 'gamma': 'scale'}
Best trial accuracy: 0.7895716945996275


In [37]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.772812,2026-04-07 16:47:04.953363,2026-04-07 16:47:07.780637,0 days 00:00:02.827274,NaN,False,RandomForest,NaN,NaN,NaN,18.0,9.0,10.0,214.0,COMPLETE
1,1,0.759777,2026-04-07 16:47:07.782804,2026-04-07 16:47:07.926378,0 days 00:00:00.143574,0.881278,NaN,SVM,auto,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
2,2,0.782123,2026-04-07 16:47:07.928824,2026-04-07 16:47:07.972885,0 days 00:00:00.044061,0.380912,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.759777,2026-04-07 16:47:07.974021,2026-04-07 16:47:09.153011,0 days 00:00:01.178990,NaN,True,RandomForest,NaN,NaN,NaN,11.0,2.0,3.0,129.0,COMPLETE
4,4,0.750466,2026-04-07 16:47:09.155959,2026-04-07 16:47:12.818218,0 days 00:00:03.662259,NaN,NaN,GradientBoosting,NaN,NaN,0.037237,16.0,6.0,8.0,166.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.772812,2026-04-07 16:47:50.076495,2026-04-07 16:47:50.121108,0 days 00:00:00.044613,0.112584,NaN,SVM,scale,sigmoid,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.785847,2026-04-07 16:47:50.122173,2026-04-07 16:47:50.156610,0 days 00:00:00.034437,0.257825,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.756052,2026-04-07 16:47:50.157627,2026-04-07 16:47:51.720594,0 days 00:00:01.562967,NaN,NaN,GradientBoosting,NaN,NaN,0.023751,11.0,3.0,7.0,85.0,COMPLETE
98,98,0.789572,2026-04-07 16:47:51.721721,2026-04-07 16:47:51.752581,0 days 00:00:00.030860,0.121889,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [38]:
study.trials_dataframe()['params_classifier'].value_counts()

,count
params_classifier,
SVM,79
GradientBoosting,11
RandomForest,10


In [39]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

,value
params_classifier,
GradientBoosting,0.749450
RandomForest,0.763128
SVM,0.777597


In [40]:
# 1. Optimization History
plot_optimization_history(study).show()

In [41]:
# 2. Slice Plot
plot_slice(study).show()

In [42]:
# 3. Hyperparameter Importance
plot_param_importances(study).show()